<center>
    <h2>Evaluation protocole — réutilisé pour le modèle Weibull AFT</h2>
</center>

---

Ce notebook reprend **tel quel** la structure du notebook officiel `Evaluation_databattle_meteorage.ipynb`
(les trois parties : génération des prédictions, sweep sur $\theta$, évaluation finale)
mais remplace les **fausses prédictions** de la Part 1 par les prédictions du modèle
**Weibull AFT** entraîné dans `weibull_final.ipynb` / `Weibull_final3km.ipynb`.

### Rappel du protocole officiel

- Un modèle émet $N^a$ prédictions $p^a_i = (s^a_i, d^a_i, \hat t^a_i)$ pour chaque alerte $a$ :
  un score de confiance, une date d'émission, et la date prédite de fin d'alerte.
- Critère **gain** : $g^a_i = t^a + 30 - \hat t^a_i$ (gain par rapport à la baseline 30 min).
- Critère **risque** : $R = M^{L3} / N^{L3}$ — nombre d'éclairs <3 km arrivant après la fin
  prédite, divisé par le total des éclairs <3 km de l'eval.
- On sélectionne la prédiction la plus précoce qui passe un seuil $\theta$ sur la confiance,
  puis on cherche le $\theta$ qui maximise le gain sous $R < R_{\text{accept}}$.

### Adaptation Weibull → score de confiance

Le protocole prend `min(predicted_date_end_alert)` parmi les prédictions passant $\theta$.
**La confidence doit donc encoder "ce CG est probablement le dernier de l'alerte"** —
sinon, en émettant une prédiction à chaque CG, la prédiction la plus précoce d'une alerte est
celle du **premier** CG (très tôt dans l'alerte) et tous les éclairs suivants sont comptés comme manqués.

**Mapping retenu** :

$$s^a_i = S_{\text{Weibull}}(30 \mid X_i) = \exp\!\left(-(30/\lambda(X_i))^k\right)$$

= probabilité prédite par le modèle qu'**aucun CG ne suive dans les 30 minutes**
(donc que ce CG soit le dernier). C'est **monotone en position dans l'alerte** :
faible au début (un autre CG est attendu), élevée à la fin.

$$\hat t^a_i = \text{date}(CG_i) + T_q^{\text{calibré}}(X_i), \qquad q = 0.95$$

Puis on applique **exactement** le code du protocole officiel (Part 2 et Part 3).

### Part 1 — Génération des prédictions à partir du modèle Weibull

On charge l'eval, on calcule les features, on prédit $T_q$ pour 11 niveaux $q$ par CG, et on
sauvegarde dans `predictions.csv` (même format que dans le notebook officiel).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import meteorage_model as mm

TRAIN_CSV = 'data(1)/data/segment_alerts_all_airports_train.csv'
EVAL_CSV  = 'segment_alerts_all_airports_eval.csv'

# --- Entraînement du Weibull AFT (identique au notebook weibull_final) ---
cg_train = mm.load_train(TRAIN_CSV)
cg_train = mm.add_features(cg_train)
cg_train = mm.add_target(cg_train, is_last_col='is_last_lightning_cloud_ground')
scaler = StandardScaler()
_, train_fit, apt_cols = mm.build_model_matrix(cg_train, scaler, [], fit_scaler=True)
aft = mm.fit_weibull(train_fit, penalizer=0.05)
scaling = mm.compute_robust_calibration(aft, train_fit)

print(f'C-index train : {aft.concordance_index_:.4f}')
print(f'Calibration   : {scaling}')

In [ ]:
# --- Chargement de l'eval + features ---
cg_eval = mm.load_eval(EVAL_CSV)
cg_eval = mm.add_features(cg_eval)
cg_eval = mm.add_target(cg_eval, is_last_col='is_last_lightning')
cg_eval = cg_eval.sort_values(['airport', 'airport_alert_id', 'date']).reset_index(drop=True)

# Matrice X (standardisée + dummies aéroport) pour TOUS les CG de l'eval
feat = cg_eval[mm.FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
X_sc = pd.DataFrame(scaler.transform(feat), columns=mm.FEATURES)
apt_dum = pd.get_dummies(cg_eval['airport'], prefix='apt', drop_first=True).astype(float)
for col in apt_cols:
    if col not in apt_dum.columns:
        apt_dum[col] = 0.0
apt_dum = apt_dum[apt_cols].reset_index(drop=True)
X = pd.concat([X_sc.reset_index(drop=True), apt_dum], axis=1)

# --- 1) Confidence = S(30 | X) = proba qu'aucun CG ne suive dans les 30 min ---
# Cette quantité est élevée pour les CG susceptibles d'être le dernier de l'alerte
# (gros gap prédit ensuite) et faible pour les CG en milieu de salve.
survival_df = aft.predict_survival_function(X, times=[30.0])
confidence = survival_df.iloc[0].values  # shape (n_cg,), valeurs dans [0, 1]

# --- 2) predicted_date_end_alert = date_CG + T_q calibré (q = 0.95) ---
Q_PRED = 0.95
Tq_raw = aft.predict_percentile(X, p=1.0 - Q_PRED).values
Tq = np.clip(scaling[Q_PRED] * Tq_raw, 0, 60)  # cap à 60 min
dates_cg = pd.to_datetime(cg_eval['date'], utc=True)
pred_end = dates_cg + pd.to_timedelta(Tq, unit='m')

predictions = pd.DataFrame({
    'airport': cg_eval['airport'].values,
    'airport_alert_id': cg_eval['airport_alert_id'].values,
    'prediction_date': dates_cg.values,
    'predicted_date_end_alert': pred_end.values,
    'confidence': confidence,
})
predictions.to_csv('predictions.csv', index=False)
print(f'Prédictions générées : {len(predictions):,} (1 par CG de l\'eval)')
print(f'Distribution confidence S(30|X) : min={confidence.min():.3f}, '
      f'med={np.median(confidence):.3f}, max={confidence.max():.3f}')
print(f'Distribution T_q (q={Q_PRED})   : min={Tq.min():.2f}m, med={np.median(Tq):.2f}m, max={Tq.max():.2f}m')
predictions.head()

### Part 2 — Évaluation : sweep sur $\theta$

**Code repris quasi-identique** au notebook officiel : on teste différentes valeurs du seuil $\theta$ sur le score de confiance, on calcule gain et nombre d'éclairs manqués, puis on plotte.

In [ ]:
import pandas as pd
from tqdm import tqdm

# notre baseline 30 minutes
MAX_GAP_MINUTES = 30

# fichier eval (= notre csv segments...eval.csv)
input_file = EVAL_CSV
df = pd.read_csv(input_file).rename(columns={'alert_id': 'airport_alert_id'})

# building the Thetas (sweep large)
n_samples = 20
thetas = [i/n_samples for i in range(n_samples)]

# min distance à laquelle un éclair est considéré dangereux
min_dist = 3
# total des éclairs dangereux dans l'eval (CG + IC, conformément au protocole)
tot_lightnings = len(df.loc[df.dist < min_dist])
print(f'Total éclairs <{min_dist} km dans eval : {tot_lightnings}')

# lecture des prédictions générées par notre modèle Weibull
predictions = pd.read_csv('predictions.csv')
predictions.predicted_date_end_alert = pd.to_datetime(predictions.predicted_date_end_alert, utc=True)

# group by alerts
alerts = df.groupby(['airport', 'airport_alert_id'])
print(f'Nombre d\'alertes dans eval : {alerts.ngroups}')

In [ ]:
# Pour chaque theta : on garde les predictions au-dessus de theta,
# on prend la plus precoce par alerte, on mesure gain + missed lightnings.
results = {}
for theta in tqdm(thetas):
    pred_over_theta = predictions.loc[predictions['confidence'] >= theta]
    pred_over_theta_min = pred_over_theta.groupby(
        ['airport', 'airport_alert_id']).predicted_date_end_alert.min()
    gain, missed_lights = 0, 0
    for (airport, alert_id), end_alert_pred in pred_over_theta_min.items():
        try:
            lightings = alerts.get_group((airport, alert_id))
        except KeyError:
            continue
        end_alert_baseline = pd.to_datetime(lightings.date, utc=True).max() + pd.Timedelta(minutes=MAX_GAP_MINUTES)
        gain += (end_alert_baseline - end_alert_pred).total_seconds()
        missed_lights += sum(pd.to_datetime(lightings.loc[lightings.dist < min_dist].date, utc=True) > end_alert_pred)
    results[theta] = (gain, missed_lights)

In [ ]:
import matplotlib.pyplot as plt
gains = [results[theta][0]/3600 for theta in thetas]
missed = [results[theta][1] / tot_lightnings for theta in thetas]
plt.figure(figsize=(10, 6))
plt.plot(missed, gains, marker='*', markersize=8)
for t, g, m in zip(thetas, gains, missed):
    plt.text(m, g, f'{t:.2f}', fontsize=8)
plt.xlabel('Proportion d\'éclairs <3km manqués')
plt.ylabel('Gain de temps (heures)')
plt.title('Gain vs risque pour différents seuils $\\theta$ — modèle Weibull AFT')
plt.grid(alpha=0.3)
plt.show()

**Sélection du meilleur $\theta$**

Le $\theta$ qui maximise le gain sous la contrainte $R < R_{\text{accept}} = 0.02$.

In [ ]:
ACCEPTABLE_RISK = 0.02

candidates = [(gain_of_time, theta, missing_rate)
              for (theta, (gain_of_time, missing_rate)) in results.items()
              if missing_rate / tot_lightnings < ACCEPTABLE_RISK]

if candidates:
    (gain_of_time, theta, missing_lightnings) = max(candidates)
    print('Le système permet de gagner',
          int(gain_of_time / 3600),
          'heures, avec un risque de', ACCEPTABLE_RISK,
          'en utilisant un seuil sur la confidence de', theta)
else:
    print(f'Aucun theta ne respecte R < {ACCEPTABLE_RISK}.')
    print('Risques observés par theta :')
    for theta, (g, m) in sorted(results.items()):
        print(f'  theta={theta:.2f}  R={m/tot_lightnings*100:.2f}%  gain={g/3600:.0f}h')

### Part 3 — Évaluation pour un $\theta$ fixé

C'est essentiellement le même code que Part 2, mais avec une seule valeur de $\theta$.
Correspond à ce que le jury lancera sur notre `predictions.csv`.

In [ ]:
# redefining some variables
import pandas as pd
predictions = pd.read_csv('predictions.csv')
predictions.predicted_date_end_alert = pd.to_datetime(predictions.predicted_date_end_alert, utc=True)
MAX_GAP_MINUTES = 30
input_file = EVAL_CSV
df = pd.read_csv(input_file).rename(columns={'alert_id': 'airport_alert_id'})
min_dist = 3
tot_lightnings = len(df.loc[df.dist < min_dist])
alerts = df.groupby(['airport', 'airport_alert_id'])

# theta obtenu de la section précédente (à mettre à jour si besoin)
theta = theta if 'theta' in dir() else 0.95

In [ ]:
# Essentiellement le même code que Part 2 du notebook officiel
pred_over_theta = predictions.loc[predictions['confidence'] >= theta]
pred_over_theta_min = pred_over_theta.groupby(
    ['airport', 'airport_alert_id']).predicted_date_end_alert.min()
gain, missed_lights = 0, 0
pred_not_there = []
for (airport, alert_id), end_alert_pred in pred_over_theta_min.items():
    try:
        lightings = alerts.get_group((airport, alert_id))
        end_alert_baseline = pd.to_datetime(lightings.date, utc=True).max() + pd.Timedelta(minutes=MAX_GAP_MINUTES)
        gain += (end_alert_baseline - end_alert_pred).total_seconds()
        missed_lights += sum(pd.to_datetime(lightings.loc[lightings.dist < min_dist].date, utc=True) > end_alert_pred)
    except Exception:
        pred_not_there.append((airport, alert_id))

print('Pour theta =', theta,
      ', le gain de temps est', int(gain / 3600),
      'heures, avec un taux d\'éclairs <3km manqués de', round(missed_lights / tot_lightnings, 4))
print(f'Alertes sans prédiction passant theta : {len(pred_not_there)}')

### Conclusion

Ce notebook **reproduit exactement** la procédure d'évaluation officielle (`Evaluation_databattle_meteorage.ipynb`) avec une seule différence : la **Part 1** ne génère pas de fausses prédictions mais utilise notre modèle **Weibull AFT** calibré.

Mapping :

| Élément du protocole         | Notre choix                                                         |
|------------------------------|---------------------------------------------------------------------|
| `prediction_date`            | date du CG observé                                                  |
| `predicted_date_end_alert`   | `date_CG` + $T_{0.95}^{\text{calibré}}(X)$                          |
| `confidence`                 | $S(30 \mid X)$ = proba prédite qu'aucun CG ne suive dans 30 min     |
| Sweep $\theta$               | $\theta \in [0, 0.05, 0.10, \dots, 0.95]$                          |
| Sélection                    | $\theta$ max gain sous $R<2\%$                                      |

Pourquoi cette `confidence` et pas $q$ directement ? Parce que le protocole prend
`min(predicted_date_end_alert)` parmi les prédictions passant $\theta$ : il faut donc
que la `confidence` permette de **filtrer les CG du milieu de l'alerte** (où le prochain CG arrive vite)
et de ne garder que les CG terminaux. $S(30 \mid X)$ encode exactement cela.

Comparer le `gain` et le `missed_lights` obtenus ici avec ceux de la section 8 de `Weibull_final3km.ipynb` permet de **vérifier la cohérence** entre l'évaluation dynamique (un point par CG) et l'évaluation officielle (une décision par alerte basée sur les prédictions filtrées par $\theta$).